# Fase 4 — Estratégia de ajustes

**Objetivo da fase:** definir como o modelo será ajustado e com quais hiperparâmetros, verificar que a configuração cabe na GPU T4 e salvá-la para o treino.

## 4.1 Preparar o ambiente

### 4.1.1 Configurar o Google Colab

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **GPU T4** e **Versão 2026.10**. Nesta fase o modelo é carregado e treinado por apenas alguns passos, só para conferir que a configuração cabe na GPU.

O secret `HF_TOKEN` criado na Fase 1 já fica disponível para todos os notebooks da conta. No painel lateral, ícone de chave, ligar *Acesso ao notebook* para o `HF_TOKEN` e rodar a célula abaixo para verificar a configuração.

In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

Conta:         poliana-moraes
Tipo de token: write


### 4.1.2 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo. O `peft` traz o LoRA, que o `SFTTrainer` do `trl` aplica sobre o modelo base:

In [3]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q transformers==5.22.0 datasets==4.8.5 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 huggingface_hub==1.33.0
# O runtime do Colab traz o torchao 0.10.0, que o peft recusa ao ser importado. O projeto não usa o torchao (não há quantização)
!pip uninstall -q -y torchao

## 4.2 Carregar o dataset conversacional

Carregar o dataset no formato prompt/completion publicado na Fase 3:

In [4]:
from datasets import load_dataset

SFT_ID = f"{conta['name']}/alertas-soc-sft"

conversas = load_dataset(SFT_ID)
print(conversas)
# Esperado: train com 1033, validation com 224 e test com 221 exemplos

README.md:   0%|          | 0.00/662 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  686kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  152kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  145kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1033 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/224 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/221 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1033
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 224
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 221
    })
})


## 4.3 Estratégia de ajuste

### 4.3.1 Comparar as estratégias

Há três formas principais de ajustar um LLM com SFT:

| Estratégia | O que é treinado | Quando usar |
|---|---|---|
| Full fine-tuning | Todos os pesos do modelo | Muitos dados e GPU grande |
| **LoRA** | Duas matrizes pequenas (A e B) somadas a algumas camadas. Os pesos originais ficam congelados | Caso geral: treina menos de 1% dos parâmetros |
| QLoRA (LoRA + 4-bit) | O mesmo que o LoRA, mas o modelo base congelado fica quantizado em 4 bits | Modelo que não cabe na GPU em `fp16` |

No LoRA, o peso ajustado de uma camada é `W + (lora_alpha / r) · B · A`. `W` é o peso original, congelado, e `B · A` é uma correção de posto `r`, a única parte treinada. Ao final, só `A` e `B` são salvos: é o **adaptador**, com poucos MB.

### 4.3.2 Estimar a memória de cada estratégia

A memória do treino depende de quantos bytes cada parâmetro ocupa:

- **Full fine-tuning** com precisão mista e AdamW: cerca de **16 bytes** por parâmetro (peso, gradiente e os dois estados do otimizador, todos em `fp32`).
- **LoRA:** **2 bytes** por parâmetro do modelo base congelado em `fp16`. O adaptador é tão pequeno que pode ser ignorado.
- **QLoRA:** cerca de **0,5 byte** por parâmetro do modelo base em 4 bits.

Estimar a memória para o modelo escolhido na Fase 1, para a alternativa e para um modelo de 7B, como referência:

In [5]:
import pandas as pd
from huggingface_hub import model_info

GB = 1e9
BYTES_POR_PARAMETRO = {"full fine-tuning": 16, "LoRA (fp16)": 2, "QLoRA (4-bit)": 0.5}

candidatos = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]

linhas = []
for model_id in candidatos:
    params = model_info(model_id).safetensors.total
    linha = {"modelo": model_id.split("/")[1], "parâmetros (bi)": params / 1e9}
    for estrategia, bytes_por_param in BYTES_POR_PARAMETRO.items():
        linha[f"{estrategia} (GB)"] = params * bytes_por_param / GB
    linhas.append(linha)

print("Memória estimada só para pesos, gradientes e otimizador (sem ativações). A T4 tem cerca de 15 GB.")
pd.DataFrame(linhas).set_index("modelo").round(1)

Memória estimada só para pesos, gradientes e otimizador (sem ativações). A T4 tem cerca de 15 GB.


,parâmetros (bi),full fine-tuning (GB),LoRA (fp16) (GB),QLoRA (4-bit) (GB)
modelo,,,,
Qwen2.5-0.5B-Instruct,0.5,7.9,1.0,0.2
Qwen2.5-1.5B-Instruct,1.5,24.7,3.1,0.8
Qwen2.5-7B-Instruct,7.6,121.8,15.2,3.8


**Conclusão**

- **0.5B:** até o full fine-tuning cabe na T4 no papel (cerca de 8 GB, mais as ativações). O problema dele aqui não é memória: com 1.033 exemplos de treino, ajustar todos os 494 milhões de pesos facilita o *overfitting* e pode apagar o que o modelo já sabe seguir de instruções. Além disso, cada versão treinada seria um modelo inteiro de cerca de 1 GB, em vez de um adaptador de poucos MB.
- **1.5B:** o full fine-tuning já não cabe, mas o LoRA em `fp16` cabe com folga. Trocar para a alternativa da Fase 1 não exige mudar de estratégia.
- **7B:** em `fp16`, só os pesos já ocupam a T4 inteira. Esse é o caso do QLoRA.

**Decisão: LoRA sobre o `Qwen2.5-0.5B-Instruct` em `fp16`.** O QLoRA não é necessário: a quantização em 4 bits economiza memória que não falta e deixa o treino mais lento e um pouco menos preciso.

### 4.3.3 Configurar o LoRA

Carregar o modelo como nas fases anteriores e listar os nomes das camadas lineares, que são as candidatas a receber o adaptador:

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

camadas_lineares = sorted({nome.split(".")[-1] for nome, m in model.named_modules() if isinstance(m, torch.nn.Linear)})
print(camadas_lineares)
print(model.model.layers[0])
# Esperado: q_proj, k_proj, v_proj e o_proj na atenção; gate_proj, up_proj e down_proj no MLP

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


Cada um dos 24 blocos do Qwen tem a **atenção** (`q_proj`, `k_proj`, `v_proj`, `o_proj`), que decide quais partes da entrada olhar, e o **MLP** (`gate_proj`, `up_proj`, `down_proj`), que transforma o que foi lido. O `lm_head` é a camada de saída e fica de fora.

| Parâmetro | O que controla | Valor | Quando mudar |
|---|---|---|---|
| `r` | Posto das matrizes A e B, ou seja, quanta capacidade o adaptador tem | `16` | Underfitting → aumentar. Overfitting → diminuir |
| `lora_alpha` | Escala da correção (`lora_alpha / r`) | `32` (escala 2) | Manter em `2 × r` ao mudar o `r` |
| `lora_dropout` | Dropout aplicado na entrada do adaptador durante o treino | `0.05` | Overfitting → aumentar para `0.1` |
| `target_modules` | Em quais camadas o adaptador é aplicado | Atenção: `q_proj`, `k_proj`, `v_proj`, `o_proj` | Underfitting → incluir o MLP |

A tarefa é estreita: aplicar uma tabela de política e devolver sempre o mesmo formato. Por isso começar só com a atenção e um `r` moderado basta. Se faltasse capacidade, incluir o MLP seria o primeiro ajuste.

Definir a configuração e conferir que os módulos alvo existem no modelo. Os nomes mudam entre arquiteturas (no Phi-3.5, por exemplo, a atenção usa `qkv_proj`), então uma configuração copiada de outro modelo pode não servir para este:

In [7]:
from peft import LoraConfig

LORA = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "task_type": "CAUSAL_LM",
}
assert set(LORA["target_modules"]) <= set(camadas_lineares), "módulo alvo não existe no modelo"

lora_config = LoraConfig(**LORA)
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'q_proj', 'o_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


## 4.4 Hiperparâmetros

### 4.4.1 Definir os valores iniciais

| Hiperparâmetro | O que controla | Valor inicial | Quando mudar |
|---|---|---|---|
| `learning_rate` | Tamanho do passo de ajuste dos pesos | `2e-4` | Loss oscila → diminuir. Loss não desce → aumentar |
| `per_device_train_batch_size` | Exemplos por passo, limitado pela memória da GPU | `8` | Falta de memória → reduzir e aumentar `gradient_accumulation_steps` |
| `gradient_accumulation_steps` | Quantos batches somar antes de atualizar os pesos | `2` (lote efetivo 16) | Compensar um batch menor |
| `num_train_epochs` | Quantas vezes o modelo vê o treino inteiro | `2` | Loss de validação sobe antes do fim → reduzir |
| `max_length` | Tamanho máximo da sequência | `256` (Fase 3) | Só se o dataset mudar |
| `eval_steps` / `logging_steps` | De quantos em quantos passos medir a validação e registrar o loss de treino | `20` / `10` | Poucos pontos na curva → diminuir |

O `learning_rate` de `2e-4` é bem maior que o usado no full fine-tuning (por volta de `2e-5`). Isso é normal no LoRA: as matrizes A e B começam do zero e são poucas, então precisam de passos maiores.

`fp16=True` liga a precisão mista. A T4 não suporta `bf16`, que o `SFTConfig` liga por padrão, então ele vai desligado de forma explícita, como na Fase 3.

In [8]:
from trl import SFTConfig

MAX_LENGTH = 256  # Fase 3, seção 3.5.2

HIPERPARAMETROS = {
    "output_dir": "outputs",
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 2,
    "max_length": MAX_LENGTH,
    "eval_strategy": "steps",
    "eval_steps": 20,
    "logging_steps": 10,
    "fp16": True,
    "bf16": False,  # o SFTConfig liga bf16 por padrão, e a T4 não suporta bf16
    "seed": 42,  # mesma semente da divisão da Fase 2
    "report_to": "tensorboard",
}

sft_config = SFTConfig(**HIPERPARAMETROS)

### 4.4.2 Calcular o número de passos

Com o lote efetivo e o tamanho do treino, dá para saber antes de treinar quantos pontos as curvas de loss da Fase 5 vão ter:

In [9]:
import math

lote_efetivo = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
passos_por_epoca = math.ceil(len(conversas["train"]) / lote_efetivo)
passos_total = passos_por_epoca * sft_config.num_train_epochs

print(f"Lote efetivo:             {lote_efetivo}")
print(f"Passos por época:         {passos_por_epoca}")
print(f"Passos no total:          {passos_total}")
print(f"Registros do loss treino: {passos_total // sft_config.logging_steps}")
print(f"Avaliações na validação:  {passos_total // sft_config.eval_steps}")
# Esperado: 65 passos por época, 130 no total e 6 avaliações

Lote efetivo:             16
Passos por época:         65
Passos no total:          130
Registros do loss treino: 13
Avaliações na validação:  6


## 4.5 Teste

Antes de gastar o tempo do treino completo, rodar **5 passos** com exatamente a mesma configuração, só mudando a duração e os registros. O teste confere três coisas:

- O lote de 8 sequências de 256 tokens cabe na memória da T4.
- O loss é um número finito (nem `NaN`, nem infinito), o que mostra que o `fp16` está estável.
- A avaliação na validação roda, porque ela só é chamada pela primeira vez no passo 20 do treino real.

O `SFTTrainer` recebe o modelo base e o `peft_config`, exatamente como na Fase 5. Ele aplica o LoRA e congela o resto:

In [10]:
from trl import SFTTrainer

config_teste = SFTConfig(**{**HIPERPARAMETROS, "max_steps": 5, "logging_steps": 1, "eval_strategy": "no", "report_to": "none"})

torch.cuda.reset_peak_memory_stats()
trainer = SFTTrainer(
    model=model,
    args=config_teste,
    train_dataset=conversas["train"],
    eval_dataset=conversas["validation"],
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()
# Esperado: cerca de 2,2 milhões de parâmetros treináveis, menos de 0,5% do total

Tokenizing train dataset:   0%|          | 0/1033 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1033 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1033 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1033 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/224 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/224 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/224 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/224 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


Treinar os 5 passos e avaliar uma vez:

In [ ]:
trainer.train()
avaliacao = trainer.evaluate()

losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
assert all(math.isfinite(l) for l in losses + [avaliacao["eval_loss"]]), "loss NaN ou infinito"

_, total = torch.cuda.mem_get_info()
print(f"Loss de treino por passo: {[round(l, 3) for l in losses]}")
print(f"Loss de validação:        {avaliacao['eval_loss']:.3f}")
print(f"Pico de memória da GPU:   {torch.cuda.max_memory_allocated() / GB:.1f} GB de {total / GB:.1f} GB")

**Leitura:** o loss de treino deve começar alto e já cair nesses poucos passos, e o pico de memória deve ficar bem abaixo do total da T4. A folga mostra que o `per_device_train_batch_size=8` pode ser mantido.

Se o teste falhar:

| Sintoma | Ação |
|---|---|
| `CUDA out of memory` | `per_device_train_batch_size=4` e `gradient_accumulation_steps=4`, mantendo o lote efetivo de 16 |
| Loss `NaN` | Diminuir o `learning_rate` e conferir que `bf16=False` |
| Loss não muda | Conferir a máscara de loss da Fase 3 e se o `print_trainable_parameters` mostra parâmetros treináveis |

O adaptador deste teste é descartado: a Fase 5 abre um runtime novo e treina do zero.

## 4.6 Salvar a configuração no Google Drive

A Fase 5 roda em outro notebook, num runtime novo. Para que ela use **exatamente** a configuração definida e testada aqui, sem copiar valores à mão, salvar o LoRA e os hiperparâmetros num JSON no Drive, na mesma pasta onde o adaptador será salvo depois.

Ao rodar a célula, o Colab pede permissão para acessar o Google Drive da conta:

In [11]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")

PROJETO_DIR = "/content/drive/MyDrive/fine-tuning-soc"
CONFIG_PATH = f"{PROJETO_DIR}/config_treino.json"
os.makedirs(PROJETO_DIR, exist_ok=True)

config_treino = {"model_id": MODEL_ID, "sft_id": SFT_ID, "lora": LORA, "sft": HIPERPARAMETROS}
with open(CONFIG_PATH, "w") as f:
    json.dump(config_treino, f, ensure_ascii=False, indent=2)

print(f"Configuração salva em {CONFIG_PATH}")

Mounted at /content/drive
Configuração salva em /content/drive/MyDrive/fine-tuning-soc/config_treino.json


Recarregar do Drive para confirmar:

In [12]:
with open(CONFIG_PATH) as f:
    config_treino = json.load(f)

lora_config = LoraConfig(**config_treino["lora"])
sft_config = SFTConfig(**config_treino["sft"])
print(json.dumps(config_treino, ensure_ascii=False, indent=2))
# Esperado: os mesmos valores das seções 4.3.3 e 4.4.1

{
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "sft_id": "poliana-moraes/alertas-soc-sft",
  "lora": {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj"
    ],
    "task_type": "CAUSAL_LM"
  },
  "sft": {
    "output_dir": "outputs",
    "learning_rate": 0.0002,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 2,
    "max_length": 256,
    "eval_strategy": "steps",
    "eval_steps": 20,
    "logging_steps": 10,
    "fp16": true,
    "bf16": false,
    "seed": 42,
    "report_to": "tensorboard"
  }
}


## 4.7 Conclusão

Configuração definida para o treino:

| Decisão | Valor | Por quê |
|---|---|---|
| Estratégia | LoRA sobre o `Qwen2.5-0.5B-Instruct` em `fp16` | Full fine-tuning arrisca *overfitting* com 1.033 exemplos. QLoRA economiza memória que não falta |
| Adaptador | `r=16`, `lora_alpha=32`, `lora_dropout=0.05`, só na atenção | Tarefa estreita, que não exige o MLP |
| Parâmetros treináveis | Cerca de 2,2 milhões, menos de 0,5% do modelo | O adaptador salvo tem poucos MB |
| Otimização | `learning_rate=2e-4`, lote efetivo 16 (8 × 2), 2 épocas | Valores usuais de LoRA para um dataset deste tamanho |
| Duração | 130 passos, loss de treino a cada 10 e validação a cada 20 | 6 pontos de validação para o diagnóstico da Fase 5 |
| Teste de fumaça | 5 passos com loss finito e avaliação executada | Configuração cabe na T4 |

**Conclusão:** o modelo será ajustado com LoRA, treinando só uma pequena fração dos parâmetros. O resto do modelo fica congelado e preserva o que ele já sabe de seguir instruções, enquanto o adaptador aprende a política de severidade e o formato da triagem. A configuração foi testada na T4 e está salva em `fine-tuning-soc/config_treino.json` no Drive. A Fase 5 carrega esse arquivo, treina os 130 passos e acompanha as curvas de loss.